# Lab 3 — Section VI Exercises

Data Mining — Classification (Decision Tree, Naive Bayes, ensemble models)

In [ ]:
import math
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


## Exercise 1 — Toy product sales (ID3, CART, Naive Bayes)

In [ ]:
# Toy dataset from lab PDF (15 records)
data = [
    {"Type": "Control", "Colors": 3, "Size": "Small", "Material": "PP Plastic", "Label": "High"},
    {"Type": "Puzzle", "Colors": 5, "Size": "Medium", "Material": "Rubber", "Label": "Low"},
    {"Type": "Puzzle", "Colors": 7, "Size": "Large", "Material": "PP Plastic", "Label": "Low"},
    {"Type": "Control", "Colors": 5, "Size": "Small", "Material": "Rubber", "Label": "Low"},
    {"Type": "Doll", "Colors": 3, "Size": "Medium", "Material": "PP Plastic", "Label": "Low"},
    {"Type": "Control", "Colors": 5, "Size": "Medium", "Material": "PP Plastic", "Label": "High"},
    {"Type": "Doll", "Colors": 5, "Size": "Large", "Material": "PP Plastic", "Label": "High"},
    {"Type": "Control", "Colors": 7, "Size": "Medium", "Material": "Rubber", "Label": "Low"},
    {"Type": "Puzzle", "Colors": 7, "Size": "Large", "Material": "Rubber", "Label": "High"},
    {"Type": "Puzzle", "Colors": 3, "Size": "Large", "Material": "PP Plastic", "Label": "Low"},
    {"Type": "Doll", "Colors": 3, "Size": "Small", "Material": "Rubber", "Label": "Low"},
    {"Type": "Puzzle", "Colors": 3, "Size": "Small", "Material": "PP Plastic", "Label": "High"},
    {"Type": "Control", "Colors": 5, "Size": "Large", "Material": "Rubber", "Label": "Low"},
    {"Type": "Doll", "Colors": 5, "Size": "Medium", "Material": "PP Plastic", "Label": "High"},
    {"Type": "Doll", "Colors": 7, "Size": "Large", "Material": "PP Plastic", "Label": "High"},
]
ATTRS = ["Type", "Colors", "Size", "Material"]
pd.DataFrame(data)


In [ ]:
def entropy(labels):
    counter = Counter(labels)
    total = len(labels)
    if total == 0:
        return 0.0
    return -sum((c / total) * math.log2(c / total) for c in counter.values())


def info_gain(data_rows, attr):
    total_entropy = entropy([d["Label"] for d in data_rows])
    values = set(d[attr] for d in data_rows)
    weighted_entropy = 0.0
    for v in values:
        subset = [d for d in data_rows if d[attr] == v]
        weighted_entropy += (len(subset) / len(data_rows)) * entropy(
            [d["Label"] for d in subset]
        )
    return total_entropy - weighted_entropy


def gini(labels):
    counter = Counter(labels)
    total = len(labels)
    if total == 0:
        return 0.0
    return 1 - sum((c / total) ** 2 for c in counter.values())


def gini_split(data_rows, attr):
    values = set(d[attr] for d in data_rows)
    g = 0.0
    for v in values:
        subset = [d for d in data_rows if d[attr] == v]
        g += (len(subset) / len(data_rows)) * gini([d["Label"] for d in subset])
    return g


def majority_label(data_rows):
    return Counter(d["Label"] for d in data_rows).most_common(1)[0][0]


def build_tree(data_rows, attrs, criterion="ig"):
    labels = [d["Label"] for d in data_rows]
    if len(set(labels)) == 1:
        return labels[0]
    if not attrs:
        return majority_label(data_rows)

    if criterion == "ig":
        scores = {a: info_gain(data_rows, a) for a in attrs}
        best = max(scores, key=scores.get)
        if scores[best] == 0:
            return majority_label(data_rows)
    else:
        scores = {a: gini_split(data_rows, a) for a in attrs}
        best = min(scores, key=scores.get)

    tree = {best: {}}
    remaining = [a for a in attrs if a != best]
    for v in set(d[best] for d in data_rows):
        subset = [d for d in data_rows if d[best] == v]
        if not subset:
            tree[best][v] = majority_label(data_rows)
        else:
            tree[best][v] = build_tree(subset, remaining, criterion=criterion)
    return tree


def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree
    attr = list(tree.keys())[0]
    value = sample[attr]
    if value in tree[attr]:
        return predict(tree[attr][value], sample)
    return "Unknown"


### Question 1 — Information Gain (ID3)

In [ ]:
print("Information Gain per attribute:")
for attr in ATTRS:
    print(f"  {attr}: {info_gain(data, attr):.4f}")

tree_id3 = build_tree(data, ATTRS, criterion="ig")
print("\nID3 decision tree:")
import pprint

pprint.pp(tree_id3)


### Question 2 — Gini index (CART)

In [ ]:
print("Weighted Gini per attribute (lower is better):")
for attr in ATTRS:
    print(f"  {attr}: {gini_split(data, attr):.4f}")

tree_cart = build_tree(data, ATTRS, criterion="gini")
print("\nCART decision tree:")
pprint.pp(tree_cart)


### Question 3 — Predict sales (ID3 tree)

In [ ]:
test_samples_ex1 = [
    {"Type": "Doll", "Colors": 3, "Size": "Large", "Material": "Rubber"},
    {"Type": "Puzzle", "Colors": 5, "Size": "Large", "Material": "PP Plastic"},
    {"Type": "Control", "Colors": 3, "Size": "Large", "Material": "Rubber"},
]

for s in test_samples_ex1:
    print(s, "->", predict(tree_id3, s))


### Question 4 — Confusion matrix, Accuracy, Recall (decision tree)

In [ ]:
y_true_ex1 = ["Low", "Low", "High"]
y_pred_tree = [predict(tree_id3, s) for s in test_samples_ex1]

cm_tree = confusion_matrix(y_true_ex1, y_pred_tree, labels=["High", "Low"])
print("Confusion matrix (rows=true, cols=pred) [High, Low]:")
print(cm_tree)
print("Accuracy:", accuracy_score(y_true_ex1, y_pred_tree))
print("Recall (pos_label=High):", recall_score(y_true_ex1, y_pred_tree, pos_label="High"))


### Question 5 — Prior P(Type = Puzzle)

In [ ]:
prob_puzzle = sum(1 for d in data if d["Type"] == "Puzzle") / len(data)
print(f"P(Type=Puzzle) = {prob_puzzle:.4f}")


### Question 6 — P(Material = Rubber | Sales = Low)

In [ ]:
subset_low = [d for d in data if d["Label"] == "Low"]
prob_rubber_given_low = sum(1 for d in subset_low if d["Material"] == "Rubber") / len(
    subset_low
)
print(f"P(Material=Rubber | Sales=Low) = {prob_rubber_given_low:.4f}")


### Questions 7–8 — Naive Bayes with Laplace smoothing

In [ ]:
label_counts = Counter(d["Label"] for d in data)


def prob(attr, value, label):
    subset = [d for d in data if d["Label"] == label]
    count = sum(1 for d in subset if d[attr] == value)
    k = len(set(d[attr] for d in data))
    return (count + 1) / (len(subset) + k)


def predict_nb(sample):
    probs = {}
    for label in ["High", "Low"]:
        p = label_counts[label] / len(data)
        for attr in sample:
            p *= prob(attr, sample[attr], label)
        probs[label] = p
    return max(probs, key=probs.get)


y_pred_nb = [predict_nb(s) for s in test_samples_ex1]
for s, pred in zip(test_samples_ex1, y_pred_nb):
    print(s, "->", pred)

cm_nb = confusion_matrix(y_true_ex1, y_pred_nb, labels=["High", "Low"])
print("\nNaive Bayes confusion matrix:")
print(cm_nb)
acc_nb_ex1 = accuracy_score(y_true_ex1, y_pred_nb)
rec_nb_ex1 = recall_score(y_true_ex1, y_pred_nb, pos_label="High")
print("Accuracy:", acc_nb_ex1)
print("Recall (pos_label=High):", rec_nb_ex1)


### Question 9 — Compare Decision Tree vs Naive Bayes

*Note: Only 3 held-out samples — metrics are illustrative, not statistically reliable.*


In [ ]:
acc_tree_ex1 = accuracy_score(y_true_ex1, y_pred_tree)
print("Decision Tree Accuracy:", acc_tree_ex1)
print("Naive Bayes Accuracy:", acc_nb_ex1)


### Question 10 — New product

In [ ]:
new_product = {
    "Type": "Puzzle",
    "Colors": 7,
    "Size": "Small",
    "Material": "Rubber",
}
print("Tree:", predict(tree_id3, new_product))
print("Naive Bayes:", predict_nb(new_product))


## Exercise 2 — Red Wine Quality (binary classification)

In [ ]:
def evaluate(y_true, y_pred, positive_label=1):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0),
    }


def metrics_df(results_dict):
    return pd.DataFrame(results_dict).T.round(4)


def run_cv(pipeline, X, y, k=5):
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
    return scores.mean(), scores.std()


def plot_feature_importance(feature_names, importances, title):
    idx = np.argsort(importances)[::-1]
    sorted_features = [feature_names[i] for i in idx]
    sorted_importances = importances[idx]
    plt.figure(figsize=(10, 5))
    sns.barplot(x=sorted_importances, y=sorted_features, orient="h")
    plt.title(title)
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


### Question 1 — Data preprocessing

In [ ]:
wine = pd.read_csv("docs/winequality-red.csv")
print("Shape:", wine.shape)
print("Missing values:\n", wine.isna().sum())
print("Duplicates:", wine.duplicated().sum())

wine = wine.drop_duplicates().reset_index(drop=True)
wine["target"] = (wine["quality"] >= 6).astype(int)
X_wine = wine.drop(columns=["quality", "target"])
y_wine = wine["target"]

print("Class distribution:\n", y_wine.value_counts())
print("Good (1) rate:", y_wine.mean())

X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=RANDOM_STATE, stratify=y_wine
)
print("Train:", X_wine_train.shape, "Test:", X_wine_test.shape)


### Questions 2–5 — Models, metrics, cross-validation

In [ ]:
models_wine = {
    "Decision Tree": Pipeline(
        [
            (
                "clf",
                DecisionTreeClassifier(
                    criterion="gini", max_depth=4, random_state=RANDOM_STATE
                ),
            )
        ]
    ),
    "Naive Bayes": Pipeline(
        [("scaler", StandardScaler()), ("clf", GaussianNB())]
    ),
    "Random Forest": Pipeline(
        [
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200, max_depth=8, random_state=RANDOM_STATE
                ),
            )
        ]
    ),
    "XGBoost": Pipeline(
        [
            (
                "clf",
                XGBClassifier(
                    n_estimators=200,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                ),
            )
        ]
    ),
}

wine_test_results = {}
fitted_wine = {}
for name, pipe in models_wine.items():
    pipe.fit(X_wine_train, y_wine_train)
    fitted_wine[name] = pipe
    y_pred = pipe.predict(X_wine_test)
    wine_test_results[name] = evaluate(y_wine_test, y_pred)

print("Wine — test set metrics:")
display(metrics_df(wine_test_results))

cv_rows = []
for name, pipe in models_wine.items():
    mean_acc, std_acc = run_cv(pipe, X_wine, y_wine, k=5)
    cv_rows.append({"model": name, "cv_mean_accuracy": mean_acc, "cv_std": std_acc})
wine_cv = pd.DataFrame(cv_rows).round(4)
print("\nWine — 5-fold CV accuracy (mean ± std):")
display(wine_cv)


### Question 3 — Which model performs better on wine?

Compare the test metrics table above. **Random Forest** and **XGBoost** usually outperform a shallow decision tree and **Gaussian Naive Bayes** on this dataset because wine features are continuous and correlated; GNB assumes conditional independence and Gaussian marginals, while tree ensembles capture non-linear interactions. The single decision tree (`max_depth=4`) underfits slightly compared to boosted/forest models.


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    fitted_wine["Decision Tree"].named_steps["clf"],
    feature_names=X_wine.columns.tolist(),
    class_names=["Bad", "Good"],
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title("Wine — Decision Tree (Gini, max_depth=4)")
plt.tight_layout()
plt.show()


### Question 6 — Feature importance (wine)

In [ ]:
for name in ["Decision Tree", "Random Forest", "XGBoost"]:
    clf = fitted_wine[name].named_steps["clf"]
    plot_feature_importance(
        X_wine.columns.tolist(),
        clf.feature_importances_,
        f"Wine — {name} feature importance",
    )


## Exercise 3 — Heart Disease dataset

### Question 1 — Load and preprocess

In [ ]:
heart = pd.read_csv("docs/heart.csv")
heart = heart.replace("?", np.nan)
print("Shape before cleaning:", heart.shape)
print("Missing:\n", heart.isna().sum())
print("Duplicates:", heart.duplicated().sum())

heart = heart.dropna().drop_duplicates().reset_index(drop=True)
RENAME = {
    "cp": "chest_pain_type",
    "trestbps": "resting_blood_pressure",
    "chol": "cholesterol",
    "fbs": "fasting_blood_sugar",
    "restecg": "rest_ecg",
    "thalach": "max_heart_rate_achieved",
    "exang": "exercise_induced_angina",
    "oldpeak": "st_depression",
    "slope": "st_slope",
    "ca": "num_major_vessels",
    "thal": "thalassemia",
}
heart = heart.rename(columns=RENAME)
print("Shape after cleaning:", heart.shape)
heart.head()


In [ ]:
X_heart = heart.drop(columns=["target"])
y_heart = heart["target"]

X_heart_train, X_heart_test, y_heart_train, y_heart_test = train_test_split(
    X_heart, y_heart, test_size=0.2, random_state=RANDOM_STATE, stratify=y_heart
)
print("Train:", X_heart_train.shape, "Test:", X_heart_test.shape)
print("Target distribution:\n", y_heart.value_counts())


### Questions 2–6 — Same pipeline as Exercise 2

In [ ]:
models_heart = {
    "Decision Tree": Pipeline(
        [
            (
                "clf",
                DecisionTreeClassifier(
                    criterion="gini", max_depth=4, random_state=RANDOM_STATE
                ),
            )
        ]
    ),
    "Naive Bayes": Pipeline(
        [("scaler", StandardScaler()), ("clf", GaussianNB())]
    ),
    "Random Forest": Pipeline(
        [
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200, max_depth=8, random_state=RANDOM_STATE
                ),
            )
        ]
    ),
    "XGBoost": Pipeline(
        [
            (
                "clf",
                XGBClassifier(
                    n_estimators=200,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                ),
            )
        ]
    ),
}

heart_test_results = {}
fitted_heart = {}
for name, pipe in models_heart.items():
    pipe.fit(X_heart_train, y_heart_train)
    fitted_heart[name] = pipe
    y_pred = pipe.predict(X_heart_test)
    heart_test_results[name] = evaluate(y_heart_test, y_pred)

print("Heart — test set metrics:")
display(metrics_df(heart_test_results))

cv_rows_h = []
for name, pipe in models_heart.items():
    mean_acc, std_acc = run_cv(pipe, X_heart, y_heart, k=5)
    cv_rows_h.append({"model": name, "cv_mean_accuracy": mean_acc, "cv_std": std_acc})
heart_cv = pd.DataFrame(cv_rows_h).round(4)
print("\nHeart — 5-fold CV accuracy:")
display(heart_cv)


### Model comparison (heart)

On heart disease data, **tree-based models** (especially Random Forest / XGBoost) often beat **GaussianNB** because features are mixed types and not jointly Gaussian. GNB can still be competitive after scaling but may misestimate boundaries for attributes like `thalassemia` and `chest_pain_type` treated as numeric.


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    fitted_heart["Decision Tree"].named_steps["clf"],
    feature_names=X_heart.columns.tolist(),
    class_names=["No disease", "Disease"],
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title("Heart — Decision Tree (Gini, max_depth=4)")
plt.tight_layout()
plt.show()

for name in ["Decision Tree", "Random Forest", "XGBoost"]:
    clf = fitted_heart[name].named_steps["clf"]
    plot_feature_importance(
        X_heart.columns.tolist(),
        clf.feature_importances_,
        f"Heart — {name} feature importance",
    )
